In [12]:
## IMPORTS AND SETUP
# Imports
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import datetime
import zipfile
import pandas as pd
import logging
import warnings
import math
import tensorflow_datasets as tfds
import gc
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tqdm.notebook import tqdm

# Force GPU usage
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("TensorFlow is using GPU: ", tf.test.is_gpu_available())
print("Devices: ", tf.config.list_physical_devices())
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        pass

# Hide TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0=default, 1=info, 2=warning, 3=error
tf.get_logger().setLevel(logging.ERROR)
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
tf.debugging.set_log_device_placement(False)
warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').setLevel(logging.FATAL)

Num GPUs Available:  1
TensorFlow is using GPU:  True
Devices:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


I0000 00:00:1744037238.887249      12 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744037238.887365      12 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744037238.887390      12 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744037238.887736      12 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-04-07 14:47:18.887771: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2112] Could not identify NUMA node of platform GPU id 0, defaulting to 0.  Your kernel may not have been built with NUMA support.

In [13]:
## PARAMETERS
seed = 123

# Dataset loading parameters
raw_data_path = "/tf/projet/Dataset"
excluded_folders = ["Dataset Livrable 2"]
delete_invalid = True
batch_size = 1000
img_height = 128
img_width = 128

# Dataset split parameters
train_split = 0.8
val_split = 0.1
test_split = 0.1

In [14]:
## DATA PREPARATION FUNCTIONS
# Data format fixes
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

def data_formats_fixes(raw_data_path):
    """
    Walks through a directory to detect and remove problematic image files.
    Removes:
    - Files that are not actually JPG format
    - Grayscale images
    - Corrupted or unreadable images
    Parameters:
    - raw_data_path: Path to the raw data folder.
    """
    print(f"Starting data format fixes...")
    print(f"Checking directory: {raw_data_path}")
    if not os.path.isdir(raw_data_path):
        print(f"Directory {raw_data_path} doesn't exist")
        return
    else:
        print(f"Directory {raw_data_path} exists")

    stats = {
        "processed": 0,
        "wrong_format_removed": 0,
        "grayscale_converted": 0,
        "corrupted_removed": 0,
        "valid_images": 0
    }

    total_files = 0
    for root, dirs, files in os.walk(raw_data_path):
        for file in files:
            if os.path.splitext(file)[1].lower() == '.jpg':
                total_files += 1

    with tqdm(total=total_files, desc="Checking images") as pbar:
        for root, dirs, files in os.walk(raw_data_path):
            for file in files:
                file_path = os.path.join(root, file)
                _, extension = os.path.splitext(file)

                if extension.lower() != '.jpg':
                    continue

                stats["processed"] += 1
                pbar.update(1)

                try:
                    with open(file_path, 'rb') as f:
                        header = f.read(4)

                    if header[:2] != b'\xff\xd8':  # Not a valid JPEG header
                        os.remove(file_path)
                        stats["wrong_format_removed"] += 1
                        continue

                    try:
                        with Image.open(file_path) as img:
                            if img.mode == 'L' or (img.mode == 'RGB' and is_grayscale(img)):
                                img_rgb = img.convert('RGB')
                                img_rgb.save(file_path, 'JPEG', quality=95)
                                stats["grayscale_converted"] += 1
                                stats["valid_images"] += 1
                                continue

                            stats["valid_images"] += 1

                    except Exception as e:
                        os.remove(file_path)
                        stats["corrupted_removed"] += 1

                except Exception as e:
                    os.remove(file_path)
                    stats["corrupted_removed"] += 1

    print(f"\nSummary:")
    print(f"Files processed: {stats['processed']}")
    print(f"Wrong format files removed: {stats['wrong_format_removed']}")
    print(f"Grayscale images converted to RGB: {stats['grayscale_converted']}")
    print(f"Corrupted files removed: {stats['corrupted_removed']}")
    print(f"Valid images remaining: {stats['valid_images']}")
    print(f"Data format fixes completed.")


# Check if an image is grayscale
def is_grayscale(img):
    """
    Check if an RGB image is actually grayscale by comparing color channels.
    Returns True if the image is grayscale.
    Parameters:
    - img: PIL Image object.
    """
    img_array = np.array(img)

    r, g, b = img_array[:,:,0], img_array[:,:,1], img_array[:,:,2]

    threshold = 5
    return (
        np.all(np.abs(r - g) <= threshold) and
        np.all(np.abs(r - b) <= threshold) and
        np.all(np.abs(g - b) <= threshold)
    )

# Dataset assembly
def dataset_assembly(raw_data_path, subfolders, batch_size, img_height, img_width, seed):
    """
    Assembles a dataset from a folder structure.
    Parameters:
    - raw_data_path: Path to the raw data folder.
    - subfolders: List of subfolders to include in the dataset.
    Returns:
    - dataset: A TensorFlow dataset object.
    """
    print(f"Starting dataset assembly...")
    try:
        dataset = tf.keras.utils.image_dataset_from_directory(
            raw_data_path,
            labels="inferred",
            label_mode="int",
            class_names=subfolders,
            color_mode="rgb",
            batch_size=batch_size,
            image_size=(img_height, img_width),
            shuffle=True,
            seed=seed,
            validation_split=None,
            subset=None,
            interpolation="bilinear",
            follow_links=False
        )

        class_names = dataset.class_names
        print(f"Detected classes: {class_names}")

        dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

        for images, labels in dataset.take(1):
            print(f"Images batch shape : {images.shape}")
            print(f"Labels batch : {labels.numpy()}")

        print(f"Dataset assembly completed.")
        return dataset, class_names

    except Exception as e:
        print(f"Error creating dataset: {e}")


# Dataset split
def dataset_split(dataset, train_split=0.8, val_split=0.1, batch_size=1000, seed=123):
    """
    Splits a dataset into training, validation, and test sets.
    Parameters:
    - dataset: The dataset to split.
    - train_split: Proportion of the dataset to use for training.
    - val_split: Proportion of the dataset to use for validation.
    - test_split: Proportion of the dataset to use for testing.
    Returns:
    - train_ds: Training dataset.
    - val_ds: Validation dataset.
    - test_ds: Test dataset.
    """
    print(f"Starting dataset split...")
    dataset_size = tf.data.experimental.cardinality(dataset).numpy()  # Faster than len(dataset)
    print(f"Dataset size: {dataset_size}")
    train_size = int(train_split * dataset_size)
    val_size = int(val_split * dataset_size)
    test_size = dataset_size - train_size - val_size

    dataset = dataset.shuffle(buffer_size=1000, seed=seed)

    print(f"Creating training set of size ~{train_size*batch_size}...")
    train_ds = dataset.take(train_size)
    remaining_ds = dataset.skip(train_size)
    print(f"Creating validation set of size ~{val_size*batch_size}...")
    val_ds = remaining_ds.take(val_size)
    print(f"Creating test set of size ~{test_size*batch_size}...")
    test_ds = remaining_ds.skip(val_size)

    print(f"Dataset split completed.")
    return train_ds, val_ds, test_ds

In [15]:
## DATA VISUALIZATION FUNCTIONS
def visualize_class_distribution_from_counts(class_counts):
    """
    Visualizes the class distribution using pre-calculated counts.
    Parameters:
    - class_counts: Dictionary with class counts
    """
    plt.figure(figsize=(10, 6))
    plt.bar(class_counts.keys(), class_counts.values())
    plt.xlabel('Classes')
    plt.ylabel('Number of samples')
    plt.title('Class distribution')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [16]:
## DATA PREPARATION WORKFLOW
# 1. Fix data formats
data_formats_fixes(raw_data_path=raw_data_path)
subfolders = [f for f in os.listdir(raw_data_path) if os.path.isdir(os.path.join(raw_data_path, f)) and f not in excluded_folders]

# 2. Assemble dataset
dataset, class_names = dataset_assembly(
    raw_data_path=raw_data_path,
    subfolders=subfolders,
    batch_size=batch_size,
    img_height=img_height,
    img_width=img_width,
    seed=seed
)

# 3. Split dataset
train_ds, val_ds, test_ds = dataset_split(
    dataset=dataset,
    train_split=train_split,
    val_split=val_split,
    batch_size=batch_size,
    seed=seed
)

Starting data format fixes...
Checking directory: /tf/projet/Dataset
Directory /tf/projet/Dataset exists


Removing grayscale: text_10000.jpg: 100%|██████████| 40557/40557 [10:34<00:00, 63.90it/s]       



Summary:
Files processed: 40557
Wrong format files removed: 23
Grayscale images removed: 11471
Corrupted files removed: 0
Valid images remaining: 29063
Data format fixes completed.
Starting dataset assembly...
Found 29863 files belonging to 5 classes.


I0000 00:00:1744037898.817890      12 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744037898.818395      12 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744037898.818424      12 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744037898.819698      12 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-04-07 14:58:18.819893: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2112] Could not identify NUMA node of platform GPU id 0, defaulting to 0.  Your kernel may not have been built with NUMA support.

Detected classes: ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']
Images batch shape : (1000, 128, 128, 3)
Labels batch : [1 1 0 0 1 1 2 4 0 1 1 2 1 2 2 2 1 0 1 1 1 1 0 2 1 2 1 0 4 0 1 2 2 1 0 1 1
 0 2 1 2 0 1 0 1 0 1 1 2 1 2 0 0 1 1 0 2 1 1 1 0 3 2 1 0 2 2 0 0 2 0 2 3 2
 1 0 0 2 4 1 0 2 3 1 2 1 0 1 2 2 0 0 1 0 2 2 1 1 1 2 0 2 2 1 2 0 0 2 0 0 2
 2 0 0 1 2 0 0 1 0 1 3 0 2 2 1 0 0 0 0 1 0 2 1 2 3 2 1 3 0 1 2 0 1 1 0 0 0
 4 2 2 2 0 0 0 0 1 2 1 0 2 2 0 3 2 2 2 0 0 1 2 2 0 1 0 1 2 0 2 2 1 0 0 2 0
 0 0 1 0 0 2 2 1 0 0 1 2 0 2 1 0 2 1 2 2 0 1 0 1 0 2 1 1 1 0 0 3 2 0 0 0 4
 2 1 2 1 0 0 0 1 0 1 3 0 2 0 2 2 2 1 1 2 2 3 2 1 2 1 1 2 1 0 1 2 1 0 0 2 1
 0 0 2 1 1 2 0 0 2 2 0 2 0 2 2 2 0 1 2 0 2 1 1 1 4 1 0 1 0 1 1 1 2 2 1 2 0
 1 1 0 1 1 1 0 2 1 1 2 2 0 0 1 2 0 0 2 2 4 1 0 0 1 1 1 0 0 1 2 0 0 0 1 0 1
 0 0 2 0 2 0 0 0 1 0 2 2 0 0 1 1 2 1 1 1 1 1 2 0 1 2 1 1 1 0 1 1 1 2 0 1 2
 1 2 2 2 1 2 0 0 0 0 0 0 0 4 1 1 0 0 1 0 0 2 2 0 2 0 2 2 0 2 2 1 2 3 0 2 3
 1 0 2 0 0 1 1 2 2 1 1 1 1 0 2 0 4 1 2 2 0 1 1 

2025-04-07 14:58:22.075489: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [17]:
## DATA VISUALIZATION WORKFLOW